# Environment / Dependencies Setup
Installs PageIndex (document tree + search), PyMuPDF (page text), langchain + OpenRouter, and dotenv.

In [ ]:
!pip install -q pageindex pymupdf langchain-core langchain-openai python-dotenv

## Import Libraries

In [ ]:
import json
import os
import re
import time
from dotenv import load_dotenv

import pymupdf
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from pageindex import PageIndexClient, utils

### Add Your Keys
Keys are read from the `.env` file next to the notebook (see the setup notes), with an `input()` fallback.

In [ ]:
# Load the secrets from the .env file next to this notebook
load_dotenv(".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Enter your OpenRouter API key (get one at https://openrouter.ai): ").strip()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
if not PAGEINDEX_API_KEY:
    PAGEINDEX_API_KEY = input("Enter your PageIndex API key (get one at https://pageindex.ai): ").strip()
os.environ["PAGEINDEX_API_KEY"] = PAGEINDEX_API_KEY

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
print("Keys loaded.")


**What you should see:** `Keys loaded.`

### Set Up the LLM
`call_llm` sends one prompt and returns the reply text. `temperature=0` keeps answers repeatable; hidden reasoning is disabled so the model doesn't think to empty.

In [ ]:
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=512,
    extra_body={"reasoning": {"enabled": False}},
)

def call_llm(prompt):
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

### Setup — Index the PDF Once
Sends the bundled PDF to PageIndex and fetches the document tree. This slow step runs once; Parts 1, 2 and 3 all reuse `doc_id` and `tree`.

In [ ]:
PDF_PATH = os.path.join("data", "CCS-Q1-2025-Earnings-Release.pdf")
if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"Could not find '{PDF_PATH}'. Run the notebook from the lab folder.")

doc_id = pi_client.submit_document(PDF_PATH)["doc_id"]
print(f"Submitted: {doc_id}")

elapsed = 0
while not pi_client.is_retrieval_ready(doc_id):
    if elapsed >= 600:
        raise TimeoutError("PageIndex timeout")
    time.sleep(5)
    elapsed += 5
print(f"Tree ready after about {elapsed}s.")

tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
utils.print_tree(tree)

**What you should see:** a tree with one root and section children (e.g. `Balance Sheet and Liquidity`), each with a summary.

### Part 1 — Reasoning Retrieval: Let the LLM Pick the Section
The LLM reads titles and summaries, picks `node_id`s, then we read those pages and answer from them — no vectors anywhere.

```mermaid
flowchart LR
    Q["Question"] --> P["LLM reads the tree<br/>(titles + summaries)"]
    P --> N["node_list + reasoning"]
    N --> M["node_id -> page numbers"]
    M --> X["Read those pages<br/>with PyMuPDF"]
    X --> A["LLM answers from<br/>that text only"]

```

In [ ]:
def parse_json(reply):
    try:
        return json.loads(reply)
    except json.JSONDecodeError:
        # DOTALL lets .* cross lines so a pretty-printed JSON object is still found
        match = re.search(r"\{.*\}", reply, re.DOTALL)
        return json.loads(match.group()) if match else {"thinking": "", "node_list": []}

def pick_sections(query):
    prompt = f"""You MUST respond with valid JSON only. No other text.

You are given a question and a document tree. Each node has: node_id, title, summary.
Pick the most specific nodes likely to contain the answer.

Question: {query}

Document tree:
{json.dumps(tree, indent=1)}

Respond with ONLY this JSON:
{{"thinking": "<your reasoning>", "node_list": ["node_id_1", "node_id_2"]}}"""
    return parse_json(call_llm(prompt))

Turn the chosen `node_id`s into page text (at most 3 pages per node, so one huge node cannot flood the prompt).

In [ ]:
node_map = utils.create_node_mapping(tree, include_page_ranges=True)

doc = pymupdf.open(PDF_PATH)
# pymupdf pages are 0-indexed but PageIndex page numbers are 1-indexed
page_texts = {i + 1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()

def build_context(node_ids, max_pages=3):
    pages = []
    for nid in node_ids:
        if nid in node_map:
            info = node_map[nid]
            span = range(info["start_index"], info["end_index"] + 1)
            # cap at max_pages per node and skip pages an earlier node already added
            pages += [p for p in list(span)[:max_pages] if p not in pages]
    context = "\n\n".join(f"--- Page {p} ---\n{page_texts[p]}" for p in pages)
    return context, pages

Finally, the whole of Part 1 in one function, then run it:

In [ ]:
def reasoning_rag(query):
    picked = pick_sections(query)
    print("Reasoning:", picked.get("thinking", "")[:300])
    for nid in picked.get("node_list", []):
        title = node_map[nid]["node"]["title"] if nid in node_map else "not in tree"
        print(f"  picked {nid} | {title}")
    context, pages = build_context(picked.get("node_list", []))
    if not context:
        return "No relevant section found.", pages
    answer = call_llm(f"""Answer using only the context. If the answer is not there, say so. Be concise.

Context:
{context}

Question: {query}""")
    return answer, pages

QUERY_1 = "How much cash and total liquidity did the company have at the end of the first quarter?"
print("Question:", QUERY_1, "\n")
answer_1, pages_1 = reasoning_rag(QUERY_1)
print("\nPages read:", pages_1)
print("Answer:", answer_1)

**What you should see:** the LLM's reasoning, a pick such as `0003 | Balance Sheet and Liquidity`, and an answer quoting the actual figures.

### Part 2 — Multi-Hop Retrieval: Collect From Several Sections
Search returns several matching sections; each visit is a *hop*, and every hop keeps a trail (title, `node_id`, pages, text).

```mermaid
flowchart LR
    Q["Question"] --> S["PageIndex search"]
    S --> H1["Hop 1: section"]
    S --> H2["Hop 2: section"]
    S --> H3["Hop 3: section"]
    H1 --> C["Combine all text"]
    H2 --> C
    H3 --> C
    C --> A["LLM answer"]
    H1 --> E["Trace: which sections,<br/>which pages, why"]
    H2 --> E
    H3 --> E

```

In [ ]:
def retrieve(query, top_k=5):
    retrieval_id = pi_client.submit_query(doc_id=doc_id, query=query).get("retrieval_id")
    if not retrieval_id:
        return []

    nodes = []
    # poll every 3s, up to 60 tries (~3 minutes), while PageIndex runs the search
    for _ in range(60):
        try:
            result = pi_client.get_retrieval(retrieval_id)
            if result.get("status") == "completed":
                nodes = result.get("retrieved_nodes", [])
                break
            if result.get("status") == "failed":
                return []
        except Exception as e:
            print(f"Temporary problem while polling, retrying... ({e})")
        time.sleep(3)

    items = []
    for i, node in enumerate(nodes[:top_k]):
        texts, pages = [], []
        for group in node.get("relevant_contents", []):
            for piece in group:
                if piece.get("relevant_content"):
                    texts.append(piece["relevant_content"])
                match = re.search(r"(\d+)", str(piece.get("physical_index", "")))
                if match and int(match.group(1)) not in pages:
                    pages.append(int(match.group(1)))
        items.append({"number": i + 1, "section": node.get("title") or f"Section {i + 1}",
                      "node_id": node.get("id", "unknown"), "pages": pages, "text": "\n".join(texts)})
    return items

`explain` prints each hop's trail: where it came from, whether the answer used it, and a short LLM-written reason.

In [ ]:
def explain(query, items, is_used, kind="Hop"):
    print("\n--- EXPLAINABILITY ---")
    for it in items:
        pages = ", ".join(map(str, it["pages"])) or "unknown"
        status = "USED in answer" if is_used(it) else "retrieved but NOT clearly used"
        print(f'\n{kind} {it["number"]}: "{it["section"]}"')
        print(f'node_id: {it["node_id"]} | page(s): {pages} | {status}')
        why = call_llm(f"""In 3-4 short lines, explain why the section below is relevant to the question.
Mention the actual numbers or facts in it. Do not repeat the question.

Question: {query}

Section title: {it["section"]}
Section content: {it["text"]}""")
        print("Why:", why)

The "was it used" check here is a rough heuristic (does the section title appear in the answer?); Part 3 replaces it with an exact check.

In [ ]:
def multi_hop_rag(query, top_k=5):
    hops = retrieve(query, top_k)
    if not hops:
        return "No relevant context found.", []
    context = "\n\n".join(h["text"] for h in hops)
    answer = call_llm(f"""You are a financial analyst. Answer using only the context below, in plain language,
the way you would explain it in a meeting. Be concise.

Context:
{context}

Question: {query}""")
    return answer, hops

QUERY_2 = ("How much did GAAP net income grow or shrink from Q1 2024 to Q1 2025? "
           "Separately, what was adjusted net income (not adjusted EBITDA) for both periods, "
           "and does it tell a different story?")
print("Question:", QUERY_2, "\n")
answer_2, hops = multi_hop_rag(QUERY_2)

print("--- SECTIONS SEARCHED ---")
for h in hops:
    print(f"Hop {h['number']}: {h['section']}")
print("\n--- ANSWER ---")
print(answer_2)

explain(QUERY_2, hops, is_used=lambda h: h["section"] in answer_2, kind="Hop")

**What you should see:** up to five hops, an answer with GAAP + adjusted income for both quarters, and a trail with one entry per hop.

### Part 3 — Table Retrieval: Whole Tables With `[Table n]` Citations
Tables come back whole and get `[Table 1]`, `[Table 2]` labels, so "was this table used?" becomes an exact tag check.

```mermaid
flowchart LR
    Q["Question"] --> R["retrieve(top_k=2)<br/>whole tables"]
    R --> L["Label: [Table 1 - title],<br/>[Table 2 - title]"]
    L --> A["LLM answers and tags<br/>every number [Table n]"]
    A --> C{"Is [Table n]<br/>in the answer?"}
    C -->|Yes| U["USED"]
    C -->|No| N["retrieved but NOT used"]

```

In [ ]:
table_prompt = """You are a financial data analyst. Answer ONLY from the text and tables below.
Pay strict attention to table rows, columns and footnotes. Do not round numbers unless asked.

CRITICAL RULES:
- Tag every number you use with its table, like this: [Table 1]
- If you use numbers from more than one table, tag each one separately.
- If the data is not in the text, say "Not found in document."

Context:
{context}

Question: {query}"""

def table_rag(query, top_k=2):
    tables = retrieve(query, top_k)
    if not tables:
        return "No relevant context found.", []
    context = "\n\n".join(f"[Table {t['number']} - {t['section']}]\n{t['text']}" for t in tables)
    return call_llm(table_prompt.format(context=context, query=query)), tables

QUERY_3 = "What was the total revenue for the three months ended March 31, 2025?"
print("Question:", QUERY_3, "\n")
answer_3, tables = table_rag(QUERY_3)

print("--- TABLES SEARCHED ---")
for t in tables:
    print(f"Table {t['number']}: {t['section']}")
print("\n--- ANSWER ---")
print(answer_3)

explain(QUERY_3, tables, is_used=lambda t: f"[Table {t['number']}]" in answer_3, kind="Table")

**What you should see:** a table-citing answer (e.g. `$903.2 million [Table 1]`) and a trail marking each one `USED` or `retrieved but NOT clearly used`.

### What We Learnt
Three retrieval styles on one PDF — reasoning, multi-hop, and whole-table — with no vector database, no embeddings, and no chunking.